In [2]:
# ---------------------------------------------------------------------
# CELL 1: SETUP
# NDVI proxy pipeline initialization
# scripts/proxies/ndvi_proxy.ipynb
# ---------------------------------------------------------------------

from pathlib import Path
from datetime import datetime, timezone
import json
import warnings

import numpy as np
import pandas as pd
import rasterio

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ---------------------------------------------------------------------
# ROOT PATHS
# ---------------------------------------------------------------------
PROJECT_ROOT = Path(r"C:\Projects\Infer RozviDrought").resolve()

DATA_DIR = PROJECT_ROOT / "data"
RASTERS_DIR = PROJECT_ROOT / "rasters"

# Observed NDVI
NDVI_OBS_DIR = DATA_DIR / "real_observations" / "ndvi"

# ERA5/common-grid climate drivers
ERA5_DIR = DATA_DIR / "simulated" / "era5_land"
ERA5_T2M_DIR = ERA5_DIR / "t2m"
ERA5_D2M_DIR = ERA5_DIR / "d2m"
ERA5_SM_DIR = ERA5_DIR / "sm"
ERA5_PET_DIR = ERA5_DIR / "pet"

# Outputs / artifacts
ARTIFACTS_DIR = DATA_DIR / "artifacts"
DATASETS_DIR = ARTIFACTS_DIR / "datasets"
MODELS_DIR = ARTIFACTS_DIR / "models"
MANIFESTS_DIR = ARTIFACTS_DIR / "manifests"

# Future corrected outputs will later go here
CORRECTED_ROOT = RASTERS_DIR / "corrected" / "cmip6"

# ---------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------
TRAIN_START = "200002"
TRAIN_END = "202602"

BACKCAST_START = "198109"
BACKCAST_END = "199912"

FUTURE_START = "202603"
FUTURE_END = "205012"

RUN_TS = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_ID = f"ndvi_proxy_{RUN_TS}"

# ---------------------------------------------------------------------
# ENSURE OUTPUT FOLDERS EXIST
# ---------------------------------------------------------------------
DATASETS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
MANIFESTS_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# HELPERS
# ---------------------------------------------------------------------
def extract_yyyymm(path: Path):
    stem = path.stem
    for token in stem.split("_"):
        if token.isdigit() and len(token) == 6:
            return token
    return None


def list_tifs(folder: Path):
    if not folder.exists():
        return []
    return sorted(folder.glob("*.tif"))


def index_by_yyyymm(folder: Path):
    out = {}
    for fp in list_tifs(folder):
        ym = extract_yyyymm(fp)
        if ym is not None:
            out[ym] = fp
    return out


def filter_range(index: dict, start_yyyymm: str, end_yyyymm: str):
    return {
        ym: fp
        for ym, fp in index.items()
        if start_yyyymm <= ym <= end_yyyymm
    }


def save_json(obj: dict, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


# ---------------------------------------------------------------------
# INDEX INPUTS
# ---------------------------------------------------------------------
ndvi_index = index_by_yyyymm(NDVI_OBS_DIR)
t2m_index = index_by_yyyymm(ERA5_T2M_DIR)
d2m_index = index_by_yyyymm(ERA5_D2M_DIR)
sm_index = index_by_yyyymm(ERA5_SM_DIR)
pet_index = index_by_yyyymm(ERA5_PET_DIR)

climate_common = set(t2m_index) & set(d2m_index) & set(sm_index) & set(pet_index)
train_overlap = sorted(set(ndvi_index) & climate_common)

# ---------------------------------------------------------------------
# STATUS PRINT
# ---------------------------------------------------------------------
print("=" * 60)
print("NDVI PROXY PIPELINE SETUP")
print("=" * 60)
print(f"PROJECT ROOT     : {PROJECT_ROOT}")
print(f"NDVI OBS DIR     : {NDVI_OBS_DIR}")
print(f"ERA5 DIR         : {ERA5_DIR}")
print(f"RUN ID           : {RUN_ID}")

print("\nINPUT STATUS")
print(f"NDVI months      : {len(ndvi_index)}")
print(f"T2M months       : {len(t2m_index)}")
print(f"D2M months       : {len(d2m_index)}")
print(f"SM months        : {len(sm_index)}")
print(f"PET months       : {len(pet_index)}")
print(f"Climate common   : {len(climate_common)}")
print(f"Train overlap    : {len(train_overlap)}")

if ndvi_index:
    print(f"NDVI first       : {min(ndvi_index.keys())}")
    print(f"NDVI last        : {max(ndvi_index.keys())}")

if train_overlap:
    print(f"Train first      : {train_overlap[0]}")
    print(f"Train last       : {train_overlap[-1]}")

print("\nTARGET WINDOWS")
print(f"Train            : {TRAIN_START} -> {TRAIN_END}")
print(f"Backcast         : {BACKCAST_START} -> {BACKCAST_END}")
print(f"Future           : {FUTURE_START} -> {FUTURE_END}")

NDVI PROXY PIPELINE SETUP
PROJECT ROOT     : C:\Projects\Infer RozviDrought
NDVI OBS DIR     : C:\Projects\Infer RozviDrought\data\real_observations\ndvi
ERA5 DIR         : C:\Projects\Infer RozviDrought\data\simulated\era5_land
RUN ID           : ndvi_proxy_20260322T125134Z

INPUT STATUS
NDVI months      : 313
T2M months       : 523
D2M months       : 523
SM months        : 555
PET months       : 523
Climate common   : 523
Train overlap    : 301
NDVI first       : 200002
NDVI last        : 202602
Train first      : 200002
Train last       : 202602

TARGET WINDOWS
Train            : 200002 -> 202602
Backcast         : 198109 -> 199912
Future           : 202603 -> 205012


In [3]:
# ---------------------------------------------------------------------
# CELL 2: SINGLE-MONTH ALIGNMENT CHECK
# Confirm one NDVI month aligns with climate drivers on common grid.
# ---------------------------------------------------------------------

from rasterio.warp import reproject, Resampling

print("=" * 60)
print("NDVI PROXY SINGLE-MONTH ALIGNMENT CHECK")
print("=" * 60)

sample_month = train_overlap[0]
print(f"Sample month : {sample_month}")

ndvi_path = ndvi_index[sample_month]
t2m_path = t2m_index[sample_month]
d2m_path = d2m_index[sample_month]
sm_path = sm_index[sample_month]
pet_path = pet_index[sample_month]

print("\nINPUT FILES")
print(f"NDVI : {ndvi_path}")
print(f"T2M  : {t2m_path}")
print(f"D2M  : {d2m_path}")
print(f"SM   : {sm_path}")
print(f"PET  : {pet_path}")

# ------------------------------------------------------------
# Read ERA5 template + drivers
# ------------------------------------------------------------
with rasterio.open(t2m_path) as ref:
    t2m = ref.read(1).astype(np.float32)
    dst_shape = (ref.height, ref.width)
    dst_transform = ref.transform
    dst_crs = ref.crs
    dst_bounds = ref.bounds

with rasterio.open(d2m_path) as src:
    d2m = src.read(1).astype(np.float32)

with rasterio.open(sm_path) as src:
    sm = src.read(1).astype(np.float32)

with rasterio.open(pet_path) as src:
    pet_raw = src.read(1).astype(np.float32)

# normalize PET sign to positive magnitude
pet = np.where(np.isfinite(pet_raw), -1.0 * pet_raw, np.nan).astype(np.float32)

# ------------------------------------------------------------
# Read NDVI and resample to ERA5 grid
# ------------------------------------------------------------
with rasterio.open(ndvi_path) as src:
    ndvi_src = src.read(1).astype(np.float32)

    ndvi_src = np.where(
        np.isfinite(ndvi_src) & (ndvi_src >= -0.2) & (ndvi_src <= 1.0),
        ndvi_src,
        np.nan
    ).astype(np.float32)

    ndvi = np.full(dst_shape, np.nan, dtype=np.float32)

    reproject(
        source=ndvi_src,
        destination=ndvi,
        src_transform=src.transform,
        src_crs=src.crs,
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        resampling=Resampling.average,
        src_nodata=np.nan,
        dst_nodata=np.nan,
    )

print("\nCOMMON GRID")
print(f"CRS        : {dst_crs}")
print(f"Shape      : {dst_shape}")
print(f"Bounds     : {dst_bounds}")

print("\nVALUE STATS")
print(f"NDVI mean  : {np.nanmean(ndvi)}")
print(f"T2M mean   : {np.nanmean(t2m)}")
print(f"D2M mean   : {np.nanmean(d2m)}")
print(f"SM mean    : {np.nanmean(sm)}")
print(f"PET mean   : {np.nanmean(pet)}")

valid_mask = (
    np.isfinite(ndvi) &
    np.isfinite(t2m) &
    np.isfinite(d2m) &
    np.isfinite(sm) &
    np.isfinite(pet)
)

print("\nVALID OVERLAP")
print(f"Valid pixels : {int(valid_mask.sum())}/{valid_mask.size}")

print("=" * 60)
print("ALIGNMENT CHECK COMPLETE")
print("=" * 60)

NDVI PROXY SINGLE-MONTH ALIGNMENT CHECK
Sample month : 200002

INPUT FILES
NDVI : C:\Projects\Infer RozviDrought\data\real_observations\ndvi\ndvi_200002.tif
T2M  : C:\Projects\Infer RozviDrought\data\simulated\era5_land\t2m\t2m_200002.tif
D2M  : C:\Projects\Infer RozviDrought\data\simulated\era5_land\d2m\d2m_200002.tif
SM   : C:\Projects\Infer RozviDrought\data\simulated\era5_land\sm\sm_200002.tif
PET  : C:\Projects\Infer RozviDrought\data\simulated\era5_land\pet\pet_200002.tif

COMMON GRID
CRS        : EPSG:4326
Shape      : (70, 80)
Bounds     : BoundingBox(left=25.100114814474466, bottom=-22.500102921341654, right=33.100151408729275, top=-15.500070901368694)

VALUE STATS
NDVI mean  : 0.5420119762420654
T2M mean   : 295.3122863769531
D2M mean   : 292.2798156738281
SM mean    : 0.38228079676628113
PET mean   : 0.00023452973982784897

VALID OVERLAP
Valid pixels : 5597/5600
ALIGNMENT CHECK COMPLETE


In [4]:
# ---------------------------------------------------------------------
# CELL 3: BUILD NDVI PROXY TRAINING DATASET
# First real proxy-building step.
# Output:
#   data/artifacts/datasets/cmip6_ndvi_proxy_training_dataset_<timestamp>.parquet
# ---------------------------------------------------------------------

from datetime import datetime, timezone
from rasterio.warp import reproject, Resampling

print("=" * 60)
print("BUILD NDVI PROXY TRAINING DATASET")
print("=" * 60)

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
NDVI_DATASET_PATH = DATASETS_DIR / f"cmip6_ndvi_proxy_training_dataset_{timestamp}.parquet"

records = []

print(f"Months to process : {len(train_overlap)}")

for month in train_overlap:
    ndvi_path = ndvi_index[month]
    t2m_path = t2m_index[month]
    d2m_path = d2m_index[month]
    sm_path = sm_index[month]
    pet_path = pet_index[month]

    # ------------------------------------------------------------
    # Read ERA5 grid + climate drivers
    # ------------------------------------------------------------
    with rasterio.open(t2m_path) as ref:
        t2m = ref.read(1).astype(np.float32)
        dst_shape = (ref.height, ref.width)
        dst_transform = ref.transform
        dst_crs = ref.crs

    with rasterio.open(d2m_path) as src:
        d2m = src.read(1).astype(np.float32)

    with rasterio.open(sm_path) as src:
        sm = src.read(1).astype(np.float32)

    with rasterio.open(pet_path) as src:
        pet_raw = src.read(1).astype(np.float32)

    # Normalize PET sign to positive magnitude
    pet = np.where(np.isfinite(pet_raw), -1.0 * pet_raw, np.nan).astype(np.float32)

    # ------------------------------------------------------------
    # Read observed NDVI and resample to ERA5 grid
    # ------------------------------------------------------------
    with rasterio.open(ndvi_path) as src:
        ndvi_src = src.read(1).astype(np.float32)

        ndvi_src = np.where(
            np.isfinite(ndvi_src) & (ndvi_src >= -0.2) & (ndvi_src <= 1.0),
            ndvi_src,
            np.nan
        ).astype(np.float32)

        ndvi = np.full(dst_shape, np.nan, dtype=np.float32)

        reproject(
            source=ndvi_src,
            destination=ndvi,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            resampling=Resampling.average,
            src_nodata=np.nan,
            dst_nodata=np.nan,
        )

    # ------------------------------------------------------------
    # Flatten into a monthly dataframe
    # ------------------------------------------------------------
    df = pd.DataFrame({
        "yyyymm": month,
        "pixel_id": np.arange(ndvi.size, dtype=np.int32),
        "t2m": t2m.flatten(),
        "d2m": d2m.flatten(),
        "sm": sm.flatten(),
        "pet": pet.flatten(),
        "ndvi_obs": ndvi.flatten(),
    })

    records.append(df)
    print(f"Processed : {month}")

# ------------------------------------------------------------
# Combine and save
# ------------------------------------------------------------
ndvi_train_df = pd.concat(records, ignore_index=True)
ndvi_train_df = ndvi_train_df.replace([np.inf, -np.inf], np.nan)

NDVI_DATASET_PATH.parent.mkdir(parents=True, exist_ok=True)
ndvi_train_df.to_parquet(NDVI_DATASET_PATH, index=False)

print("\n" + "=" * 60)
print("NDVI TRAINING DATASET SAVED")
print("=" * 60)
print(f"Rows    : {len(ndvi_train_df)}")
print(f"Path    : {NDVI_DATASET_PATH}")
print(f"Columns : {list(ndvi_train_df.columns)}")

BUILD NDVI PROXY TRAINING DATASET
Months to process : 301
Processed : 200002
Processed : 200003
Processed : 200004
Processed : 200005
Processed : 200006
Processed : 200007
Processed : 200008
Processed : 200009
Processed : 200010
Processed : 200011
Processed : 200012
Processed : 200101
Processed : 200102
Processed : 200103
Processed : 200104
Processed : 200105
Processed : 200106
Processed : 200107
Processed : 200108
Processed : 200109
Processed : 200110
Processed : 200111
Processed : 200112
Processed : 200201
Processed : 200202
Processed : 200203
Processed : 200204
Processed : 200205
Processed : 200206
Processed : 200207
Processed : 200208
Processed : 200209
Processed : 200210
Processed : 200211
Processed : 200212
Processed : 200301
Processed : 200302
Processed : 200303
Processed : 200304
Processed : 200305
Processed : 200306
Processed : 200307
Processed : 200308
Processed : 200309
Processed : 200310
Processed : 200311
Processed : 200312
Processed : 200401
Processed : 200402
Processed :

In [5]:
# ---------------------------------------------------------------------
# CELL: SAVE NDVI TRAINING DATASET MANIFEST
# ---------------------------------------------------------------------

from datetime import datetime, timezone
import json

print("=" * 60)
print("SAVE NDVI TRAINING DATASET MANIFEST")
print("=" * 60)

manifest_ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

manifest = {
    "run_id": f"ndvi_proxy_dataset_{manifest_ts}",
    "dataset_path": str(NDVI_DATASET_PATH),
    "rows": int(len(ndvi_train_df)),
    "columns": list(ndvi_train_df.columns),

    "target_variable": "ndvi_obs",
    "predictors": ["t2m", "d2m", "sm", "pet"],

    "train_period": {
        "start": TRAIN_START,
        "end": TRAIN_END,
        "months": len(train_overlap),
    },

    "gap_periods": {
        "backcast": {
            "start": BACKCAST_START,
            "end": BACKCAST_END,
        },
        "future": {
            "start": FUTURE_START,
            "end": FUTURE_END,
        },
    },

    "grid": {
        "crs": "EPSG:4326",
        "shape": [70, 80],
        "pixel_count": 70 * 80,
    },

    "created_utc": manifest_ts,
}

manifest_path = MANIFESTS_DIR / f"ndvi_proxy_dataset_{manifest_ts}_manifest.json"

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("\n" + "=" * 60)
print("MANIFEST SAVED")
print("=" * 60)
print(f"Path : {manifest_path}")
print(f"Rows : {manifest['rows']}")

SAVE NDVI TRAINING DATASET MANIFEST

MANIFEST SAVED
Path : C:\Projects\Infer RozviDrought\data\artifacts\manifests\ndvi_proxy_dataset_20260322T130201Z_manifest.json
Rows : 1685600


In [2]:
from pathlib import Path
import rasterio
import numpy as np

print("=" * 60)
print("NDVI FULL OUTPUT VALIDATION")
print("=" * 60)

PROJECT_ROOT = Path(r"C:\Projects\Infer RozviDrought")

NDVI_OUT_DIR = (
    PROJECT_ROOT
    / "rasters"
    / "corrected"
    / "proxies"
    / "ndvi"
)

SCENARIOS = ["ssp245", "ssp370", "ssp585"]

EXPECTED_HIST_START = "198109"
EXPECTED_HIST_END   = "202602"

EXPECTED_FUTURE_START = "202603"
EXPECTED_FUTURE_END   = "205012"


def extract_yyyymm(path):
    stem = path.stem
    for token in stem.split("_"):
        if token.isdigit() and len(token) == 6:
            return token
    return None


def month_range(start, end):
    months = []
    y = int(start[:4])
    m = int(start[4:])

    while True:
        ym = f"{y:04d}{m:02d}"
        months.append(ym)

        if ym == end:
            break

        m += 1
        if m > 12:
            m = 1
            y += 1

    return months


def validate_folder(folder, start, end):
    files = sorted(folder.glob("*.tif"))

    months_found = sorted(
        extract_yyyymm(fp)
        for fp in files
        if extract_yyyymm(fp) is not None
    )

    expected_months = month_range(start, end)

    missing = sorted(set(expected_months) - set(months_found))

    sample = files[0] if files else None

    stats = None

    if sample is not None:
        with rasterio.open(sample) as src:
            arr = src.read(1)

            stats = {
                "crs": src.crs.to_string(),
                "shape": arr.shape,
                "min": float(np.nanmin(arr)),
                "max": float(np.nanmax(arr)),
                "mean": float(np.nanmean(arr)),
            }

    return {
        "months_found": len(months_found),
        "first_month": months_found[0] if months_found else None,
        "last_month": months_found[-1] if months_found else None,
        "missing_months": missing[:5],
        "missing_count": len(missing),
        "sample_stats": stats,
    }


# ------------------------------------------------------------
# HISTORICAL
# ------------------------------------------------------------
print("\nHISTORICAL NDVI")
print("-" * 60)

hist_dir = NDVI_OUT_DIR / "historical"

hist_result = validate_folder(
    hist_dir,
    EXPECTED_HIST_START,
    EXPECTED_HIST_END,
)

print(hist_result)


# ------------------------------------------------------------
# FUTURE
# ------------------------------------------------------------
print("\nFUTURE NDVI")
print("-" * 60)

for scenario in SCENARIOS:

    scen_dir = NDVI_OUT_DIR / scenario

    result = validate_folder(
        scen_dir,
        EXPECTED_FUTURE_START,
        EXPECTED_FUTURE_END,
    )

    print(
        {
            "scenario": scenario,
            **result
        }
    )


print("\n" + "=" * 60)
print("NDVI VALIDATION COMPLETE")
print("=" * 60)

NDVI FULL OUTPUT VALIDATION

HISTORICAL NDVI
------------------------------------------------------------
{'months_found': 534, 'first_month': '198109', 'last_month': '202602', 'missing_months': [], 'missing_count': 0, 'sample_stats': {'crs': 'EPSG:4326', 'shape': (70, 80), 'min': -0.12096705287694931, 'max': 0.8261405229568481, 'mean': 0.5420119762420654}}

FUTURE NDVI
------------------------------------------------------------
{'scenario': 'ssp245', 'months_found': 298, 'first_month': '202603', 'last_month': '205012', 'missing_months': [], 'missing_count': 0, 'sample_stats': {'crs': 'EPSG:4326', 'shape': (70, 80), 'min': 0.5972163081169128, 'max': 0.6909723281860352, 'mean': 0.6540781855583191}}
{'scenario': 'ssp370', 'months_found': 298, 'first_month': '202603', 'last_month': '205012', 'missing_months': [], 'missing_count': 0, 'sample_stats': {'crs': 'EPSG:4326', 'shape': (70, 80), 'min': 0.5972163081169128, 'max': 0.686848521232605, 'mean': 0.6544253826141357}}
{'scenario': 'ssp58